# IPL Win Probability — Multi-Model Training Pipeline

This notebook trains 5 models (Logistic Regression, Random Forest, XGBoost, BiLSTM, GRU) on IPL ball-by-ball data and evaluates them on the 2023 holdout season.

**Validation**: Expanding-window temporal CV (3 folds)  
**Final Model**: Retrained on all data ≤ 2022 with best hyperparameters  
**Hardware**: CUDA GPU 


## 1. Setup & Dependencies

In [ ]:
!pip install -q xgboost pyyaml scikit-learn pandas numpy torch

## 2. Upload Data

Upload `train_data.csv` and `test_data.csv` from your `dataset/` folder.

In [ ]:
from google.colab import files
import os

os.makedirs('dataset', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Upload train and test data
print("Upload train_data.csv:")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'dataset/{name}', 'wb') as f:
        f.write(data)

print("\nUpload test_data.csv:")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'dataset/{name}', 'wb') as f:
        f.write(data)

print("\nFiles in dataset/:", os.listdir('dataset'))

## 3. Shared Utilities

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ─── Constants ─────────────────────────────────────────────────────
EXCLUDE_COLS = ['match_id', 'ball_idx', 'season', 'is_batting_team_winner']
TARGET_COL = 'is_batting_team_winner'

TEMPORAL_CV_FOLDS = [
    (2019, 2020),
    (2020, 2021),
    (2021, 2022),
]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ─── Data Loading ─────────────────────────────────────────────────
def load_data(subset='train'):
    df = pd.read_csv(f'dataset/{subset}_data.csv')\n
    df = df.fillna(0)\n
    return df\n

def get_features(df):
    return [c for c in df.columns if c not in EXCLUDE_COLS]

df_train = load_data('train')
df_test = load_data('test')
features = get_features(df_train)
print(f"Train rows: {len(df_train)}, Test rows: {len(df_test)}")
print(f"Features ({len(features)}): {features}")

# ─── Results Tracker ──────────────────────────────────────────────
results = {}

## 4. Logistic Regression

In [ ]:
lr_configs = [
    {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000},
    {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000},
    {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000},
    {'C': 10.0, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 1000},
    {'C': 0.1, 'penalty': 'l1', 'solver': 'saga', 'max_iter': 2000},
    {'C': 1.0, 'penalty': 'l1', 'solver': 'saga', 'max_iter': 2000},
    {'C': 10.0, 'penalty': 'l1', 'solver': 'saga', 'max_iter': 2000},
    {'C': 1.0, 'penalty': 'elasticnet', 'solver': 'saga', 'l1_ratio': 0.5, 'max_iter': 2000},
]

best_f1, best_cfg = -1, None

for i, cfg in enumerate(lr_configs):
    print(f"[Config {i+1}/{len(lr_configs)}] {cfg}")
    fold_scores = []
    for train_end, val_season in TEMPORAL_CV_FOLDS:
        tr = df_train[df_train['season'] <= train_end]
        vl = df_train[df_train['season'] == val_season]
        model = make_pipeline(StandardScaler(), LogisticRegression(**cfg))
        model.fit(tr[features], tr[TARGET_COL])
        f1 = f1_score(vl[TARGET_COL], model.predict(vl[features]))
        fold_scores.append(f1)
        print(f"  Fold (≤{train_end}/{val_season}): F1={f1:.4f}")
    mean_f1 = np.mean(fold_scores)
    print(f"  → Mean F1 = {mean_f1:.4f}\n")
    if mean_f1 > best_f1:
        best_f1, best_cfg = mean_f1, cfg

print(f"Best LR Config: {best_cfg} (F1={best_f1:.4f})")

# Retrain on all data
lr_model = make_pipeline(StandardScaler(), LogisticRegression(**best_cfg))
lr_model.fit(df_train[features], df_train[TARGET_COL])
joblib.dump(lr_model, 'models/logistic_regression.pkl')

preds = lr_model.predict(df_test[features])
results['Logistic Regression'] = {
    'Accuracy': accuracy_score(df_test[TARGET_COL], preds),
    'Precision': precision_score(df_test[TARGET_COL], preds),
    'Recall': recall_score(df_test[TARGET_COL], preds),
    'F1': f1_score(df_test[TARGET_COL], preds),
}
print(f"\nTest Results: {results['Logistic Regression']}")

## 5. Random Forest

In [ ]:
rf_configs = [
    {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    {'n_estimators': 200, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2'},
    {'n_estimators': 500, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None},
]

best_f1, best_cfg = -1, None

for i, cfg in enumerate(rf_configs):
    print(f"[Config {i+1}/{len(rf_configs)}] {cfg}")
    fold_scores = []
    for train_end, val_season in TEMPORAL_CV_FOLDS:
        tr = df_train[df_train['season'] <= train_end]
        vl = df_train[df_train['season'] == val_season]
        model = RandomForestClassifier(random_state=42, n_jobs=-1, **cfg)
        model.fit(tr[features], tr[TARGET_COL])
        f1 = f1_score(vl[TARGET_COL], model.predict(vl[features]))
        fold_scores.append(f1)
        print(f"  Fold (≤{train_end}/{val_season}): F1={f1:.4f}")
    mean_f1 = np.mean(fold_scores)
    print(f"  → Mean F1 = {mean_f1:.4f}\n")
    if mean_f1 > best_f1:
        best_f1, best_cfg = mean_f1, cfg

print(f"Best RF Config: {best_cfg} (F1={best_f1:.4f})")

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, **best_cfg)
rf_model.fit(df_train[features], df_train[TARGET_COL])
joblib.dump(rf_model, 'models/random_forest.pkl')

preds = rf_model.predict(df_test[features])
results['Random Forest'] = {
    'Accuracy': accuracy_score(df_test[TARGET_COL], preds),
    'Precision': precision_score(df_test[TARGET_COL], preds),
    'Recall': recall_score(df_test[TARGET_COL], preds),
    'F1': f1_score(df_test[TARGET_COL], preds),
}
print(f"\nTest Results: {results['Random Forest']}")

## 6. XGBoost

In [ ]:
xgb_configs = [
    {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0},
    {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0},
    {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.1, 'reg_lambda': 1.0},
    {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_alpha': 0.0, 'reg_lambda': 2.0},
    {'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0},
    {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_alpha': 0.5, 'reg_lambda': 1.5},
]

best_f1, best_cfg = -1, None

for i, cfg in enumerate(xgb_configs):
    print(f"[Config {i+1}/{len(xgb_configs)}] {cfg}")
    fold_scores = []
    for train_end, val_season in TEMPORAL_CV_FOLDS:
        tr = df_train[df_train['season'] <= train_end]
        vl = df_train[df_train['season'] == val_season]
        model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0, **cfg)
        model.fit(tr[features], tr[TARGET_COL])
        f1 = f1_score(vl[TARGET_COL], model.predict(vl[features]))
        fold_scores.append(f1)
        print(f"  Fold (≤{train_end}/{val_season}): F1={f1:.4f}")
    mean_f1 = np.mean(fold_scores)
    print(f"  → Mean F1 = {mean_f1:.4f}\n")
    if mean_f1 > best_f1:
        best_f1, best_cfg = mean_f1, cfg

print(f"Best XGB Config: {best_cfg} (F1={best_f1:.4f})")

xgb_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0, **best_cfg)
xgb_model.fit(df_train[features], df_train[TARGET_COL])
joblib.dump(xgb_model, 'models/xgboost.pkl')

preds = xgb_model.predict(df_test[features])
results['XGBoost'] = {
    'Accuracy': accuracy_score(df_test[TARGET_COL], preds),
    'Precision': precision_score(df_test[TARGET_COL], preds),
    'Recall': recall_score(df_test[TARGET_COL], preds),
    'F1': f1_score(df_test[TARGET_COL], preds),
}
print(f"\nTest Results: {results['XGBoost']}")

## 7. Shared Sequence Utilities (BiLSTM & GRU)

In [ ]:
class MatchSequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), torch.FloatTensor([self.labels[idx]])

def build_sequences(df, features, seq_length, scaler=None):
    if scaler is None:
        scaler = StandardScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df[features]), columns=features, index=df.index)
    else:
        df_scaled = pd.DataFrame(scaler.transform(df[features]), columns=features, index=df.index)

    sequences, labels = [], []
    for (mid, inn), group in df.groupby(['match_id', 'innings']):
        seq = df_scaled.loc[group.index].values
        label = group[TARGET_COL].iloc[-1]
        if len(seq) >= seq_length:
            seq = seq[-seq_length:]
        else:
            pad = np.zeros((seq_length - len(seq), len(features)))
            seq = np.vstack([pad, seq])
        sequences.append(seq)
        labels.append(int(label))
    return sequences, labels, scaler

def train_one_epoch(model, dl, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X, y in dl:
        X, y = X.to(device), y.to(device).squeeze()
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X)
    return total_loss / len(dl.dataset)

def eval_model(model, dl, device):
    model.eval()
    preds, lbls = [], []
    with torch.no_grad():
        for X, y in dl:
            logits = model(X.to(device))
            preds.extend((torch.sigmoid(logits) >= 0.5).cpu().numpy().astype(int))
            lbls.extend(y.squeeze().numpy().astype(int))
    return f1_score(lbls, preds)

def run_sequence_search(model_class, configs, model_name):
    best_f1, best_cfg = -1, None
    for i, cfg in enumerate(configs):
        print(f"\n[Config {i+1}/{len(configs)}] {cfg}")
        seq_len = cfg.get('sequence_length', 120)
        fold_scores = []
        for train_end, val_season in TEMPORAL_CV_FOLDS:
            tr = df_train[df_train['season'] <= train_end]
            vl = df_train[df_train['season'] == val_season]
            tr_seqs, tr_lbl, scaler = build_sequences(tr, features, seq_len)
            vl_seqs, vl_lbl, _ = build_sequences(vl, features, seq_len, scaler)
            tr_dl = DataLoader(MatchSequenceDataset(tr_seqs, tr_lbl), batch_size=cfg['batch_size'], shuffle=True)
            vl_dl = DataLoader(MatchSequenceDataset(vl_seqs, vl_lbl), batch_size=cfg['batch_size'])
            model = model_class(len(features), cfg['hidden_dim'], cfg['num_layers'], cfg['dropout']).to(DEVICE)
            opt = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'])
            crit = nn.BCEWithLogitsLoss()
            best_vf1, patience_ctr = 0, 0
            for ep in range(cfg['epochs']):
                loss = train_one_epoch(model, tr_dl, opt, crit, DEVICE)
                vf1 = eval_model(model, vl_dl, DEVICE)
                if vf1 > best_vf1: best_vf1, patience_ctr = vf1, 0
                else: patience_ctr += 1
                if (ep+1) % 5 == 0: print(f"    Ep {ep+1}: loss={loss:.4f} val_f1={vf1:.4f}")
                if patience_ctr >= 5: print(f"    Early stop ep {ep+1}"); break
            fold_scores.append(best_vf1)
            print(f"  Fold (≤{train_end}/{val_season}): F1={best_vf1:.4f}")
        mean_f1 = np.mean(fold_scores)
        print(f"  → Mean F1 = {mean_f1:.4f}")
        if mean_f1 > best_f1: best_f1, best_cfg = mean_f1, cfg

    print(f"\nBest {model_name} Config: {best_cfg} (F1={best_f1:.4f})")

    # Retrain on all data
    seq_len = best_cfg.get('sequence_length', 120)
    tr_seqs, tr_lbl, scaler = build_sequences(df_train, features, seq_len)
    tr_dl = DataLoader(MatchSequenceDataset(tr_seqs, tr_lbl), batch_size=best_cfg['batch_size'], shuffle=True)
    final = model_class(len(features), best_cfg['hidden_dim'], best_cfg['num_layers'], best_cfg['dropout']).to(DEVICE)
    opt = torch.optim.Adam(final.parameters(), lr=best_cfg['learning_rate'])
    crit = nn.BCEWithLogitsLoss()
    for ep in range(best_cfg['epochs']):
        loss = train_one_epoch(final, tr_dl, opt, crit, DEVICE)
        if (ep+1) % 5 == 0: print(f"  Ep {ep+1}: loss={loss:.4f}")

    torch.save(final.state_dict(), f'models/{model_name}.pth')
    joblib.dump(scaler, f'models/{model_name}_scaler.pkl')

    # Test
    te_seqs, te_lbl, _ = build_sequences(df_test, features, seq_len, scaler)
    te_dl = DataLoader(MatchSequenceDataset(te_seqs, te_lbl), batch_size=best_cfg['batch_size'])
    final.eval()
    preds, lbls = [], []
    with torch.no_grad():
        for X, y in te_dl:
            logits = final(X.to(DEVICE))
            preds.extend((torch.sigmoid(logits) >= 0.5).cpu().numpy().astype(int))
            lbls.extend(y.squeeze().numpy().astype(int))

    results[model_name.upper()] = {
        'Accuracy': accuracy_score(lbls, preds),
        'Precision': precision_score(lbls, preds),
        'Recall': recall_score(lbls, preds),
        'F1': f1_score(lbls, preds),
    }
    print(f"Test Results: {results[model_name.upper()]}")
    return final, scaler

print("Sequence utilities ready.")

## 8. BiLSTM

In [ ]:
class IPLBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                           bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :])).squeeze(-1)

bilstm_configs = [
    {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 30, 'sequence_length': 120},
    {'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 30, 'sequence_length': 120},
    {'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 40, 'sequence_length': 150},
    {'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.4, 'learning_rate': 0.001, 'batch_size': 128, 'epochs': 25, 'sequence_length': 120},
    {'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.0005, 'batch_size': 64, 'epochs': 30, 'sequence_length': 150},
]

bilstm_model, bilstm_scaler = run_sequence_search(IPLBiLSTM, bilstm_configs, 'bilstm')

## 9. GRU

In [ ]:
class IPLGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(self.dropout(out[:, -1, :])).squeeze(-1)

gru_configs = [
    {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 30, 'sequence_length': 120},
    {'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 30, 'sequence_length': 120},
    {'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.0005, 'batch_size': 32, 'epochs': 40, 'sequence_length': 150},
    {'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.4, 'learning_rate': 0.001, 'batch_size': 128, 'epochs': 25, 'sequence_length': 120},
    {'hidden_dim': 256, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.0005, 'batch_size': 64, 'epochs': 30, 'sequence_length': 150},
]

gru_model, gru_scaler = run_sequence_search(IPLGRU, gru_configs, 'gru')

## 10. Comparative Results

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
results_df.index.name = 'Model'
print("\n" + "="*60)
print("COMPARATIVE RESULTS — 2023 Holdout")
print("="*60)
print(results_df.to_string())
print("="*60)

## 11. Download Trained Models

In [ ]:
import shutil
shutil.make_archive('trained_models', 'zip', 'models')
files.download('trained_models.zip')
print("Download started! Extract the zip into your local models/ folder.")